# Sepsis Prediction (6-12h Horizon)

Predict whether a patient will develop sepsis within the next 6-12 ICU hours.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, classification_report, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
# Config
ROOT = Path.cwd().resolve()
if ROOT.name != "safe_dss":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "preprocessing" / "imputed_dataset.csv"
OUT_DIR = ROOT / "analysis" / "outputs" / "sepsis_6_12h"
EARLY_WARNING_ONLY = True

print("Data:", DATA_PATH)
print("Output:", OUT_DIR)

In [ ]:
def load_and_clean(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    for c in df.columns:
        if c == "patient_id":
            continue
        df[c] = pd.to_numeric(df[c], errors="coerce")

    if "EtCO2" in df.columns and df["EtCO2"].isna().all():
        df = df.drop(columns=["EtCO2"])

    for c in ("Unit1", "Unit2"):
        if c in df.columns and df[c].isna().any():
            df[c] = df[c].fillna(df[c].median())

    return df


def add_pattern_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["patient_id", "ICULOS"]).copy()
    g = df.groupby("patient_id", sort=False)

    roll_cols = ["MAP", "HR", "Lactate", "Resp", "WBC", "Temp", "Creatinine", "Glucose"]
    roll_cols = [c for c in roll_cols if c in df.columns]

    for c in roll_cols:
        df[f"{c}_roll3"] = g[c].transform(lambda s: s.rolling(window=3, min_periods=1).mean())

    for c in ["MAP", "Lactate", "HR"]:
        if c in df.columns:
            df[f"{c}_delta1"] = g[c].diff().fillna(0.0)

    return df


def add_horizon_labels(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["patient_id", "ICULOS"]).copy()
    g = df.groupby("patient_id", sort=False)["SepsisLabel"]

    fut = pd.concat([g.shift(-k).rename(f"_f{k}") for k in range(6, 13)], axis=1)
    df["y_onset_6_12h"] = fut.max(axis=1, skipna=False)
    df["_future_hours_ok"] = fut.notna().all(axis=1)
    return df


def build_matrix(df: pd.DataFrame):
    drop_cols = {"patient_id", "SepsisLabel", "y_onset_6_12h", "_future_hours_ok"}
    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    y = df["y_onset_6_12h"].astype(int).values
    groups = df["patient_id"].values
    return X, y, groups

In [ ]:
df = load_and_clean(DATA_PATH)
df = add_pattern_features(df)
df = add_horizon_labels(df)
df = df[df["_future_hours_ok"]].copy()

if EARLY_WARNING_ONLY:
    df = df[df["SepsisLabel"] == 0].copy()
    cohort_note = "early_warning_cohort (SepsisLabel == 0 at t)"
else:
    cohort_note = "all rows with full future window"

X, y, groups = build_matrix(df)
feature_names = list(X.columns)

print("Cohort:", cohort_note)
print(f"Rows: {len(df):,}")
print(f"Positive rate (y): {y.mean():.4f}")
print("N features:", len(feature_names))

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
fold_metrics = []
coef_accum = []

for fold, (tr, va) in enumerate(sgkf.split(X, y, groups)):
    X_tr, X_va = X.iloc[tr], X.iloc[va]
    y_tr, y_va = y[tr], y[va]

    pre = ColumnTransformer([
        ("num", StandardScaler(), feature_names),
    ], remainder="drop")

    logit = Pipeline([
        ("prep", pre),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs")),
    ])
    logit.fit(X_tr, y_tr)
    p_logit = logit.predict_proba(X_va)[:, 1]

    fold_metrics.append({
        "fold": fold,
        "model": "logistic_l2",
        "auroc": float(roc_auc_score(y_va, p_logit)),
        "auprc": float(average_precision_score(y_va, p_logit)),
    })
    coef_accum.append(np.abs(logit.named_steps["clf"].coef_.ravel()))

    hgb = HistGradientBoostingClassifier(
        max_depth=6,
        max_iter=200,
        learning_rate=0.06,
        random_state=42,
        class_weight="balanced",
    )
    hgb.fit(X_tr, y_tr)
    p_hgb = hgb.predict_proba(X_va)[:, 1]

    fold_metrics.append({
        "fold": fold,
        "model": "hist_gbrt",
        "auroc": float(roc_auc_score(y_va, p_hgb)),
        "auprc": float(average_precision_score(y_va, p_hgb)),
    })

metrics_df = pd.DataFrame(fold_metrics)
metrics_summary = metrics_df.groupby("model")[["auroc", "auprc"]].agg(["mean", "std"])
metrics_summary

In [ ]:
# Final holdout split for reporting + importance
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_tr, idx_te = next(gss.split(X, y, groups))

X_tr, X_te = X.iloc[idx_tr], X.iloc[idx_te]
y_tr, y_te = y[idx_tr], y[idx_te]

pre = ColumnTransformer([
    ("num", StandardScaler(), feature_names),
], remainder="drop")

final_logit = Pipeline([
    ("prep", pre),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs")),
])
final_logit.fit(X_tr, y_tr)
p_logit_te = final_logit.predict_proba(X_te)[:, 1]

hgb_final = HistGradientBoostingClassifier(
    max_depth=6,
    max_iter=200,
    learning_rate=0.06,
    random_state=42,
    class_weight="balanced",
)
hgb_final.fit(X_tr, y_tr)
p_hgb_te = hgb_final.predict_proba(X_te)[:, 1]

print("Holdout Logistic AUROC:", round(roc_auc_score(y_te, p_logit_te), 4))
print("Holdout Logistic AUPRC:", round(average_precision_score(y_te, p_logit_te), 4))
print("Holdout HGB AUROC:", round(roc_auc_score(y_te, p_hgb_te), 4))
print("Holdout HGB AUPRC:", round(average_precision_score(y_te, p_hgb_te), 4))

In [ ]:
# Importance outputs
coef_mean = np.mean(coef_accum, axis=0)
logit_imp = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_coef_cv": coef_mean,
}).sort_values("mean_abs_coef_cv", ascending=False)

rng = np.random.RandomState(42)
n_perm = min(8000, len(X_te))
perm_pos = rng.choice(len(X_te), size=n_perm, replace=False)
X_perm = X_te.iloc[perm_pos]
y_perm = y_te[perm_pos]

perm_res = permutation_importance(
    hgb_final,
    X_perm,
    y_perm,
    n_repeats=5,
    random_state=42,
    scoring="roc_auc",
    n_jobs=1,
)
perm_imp = pd.DataFrame({
    "feature": feature_names,
    "delta_roc_auc_mean": perm_res.importances_mean,
}).sort_values("delta_roc_auc_mean", ascending=False)

# Plot top features instead of only listing tables
k = 15
plot_logit = logit_imp.head(k).sort_values("mean_abs_coef_cv", ascending=True)
plot_perm = perm_imp.head(k).sort_values("delta_roc_auc_mean", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 8), constrained_layout=True)

axes[0].barh(plot_logit["feature"], plot_logit["mean_abs_coef_cv"], color="#1f77b4")
axes[0].set_title("Logistic importance (|coef|, CV mean)")
axes[0].set_xlabel("Mean absolute coefficient")
axes[0].set_ylabel("Feature")

axes[1].barh(plot_perm["feature"], plot_perm["delta_roc_auc_mean"], color="#ff7f0e")
axes[1].set_title("HGB permutation importance")
axes[1].set_xlabel("Mean AUROC drop after permutation")
axes[1].set_ylabel("Feature")

plt.show()

print("Top logistic features (table):")
display(logit_imp.head(k))
print("Top HGB permutation features (table):")
display(perm_imp.head(k))

### b(iii) Patient profiles (interpretation from Stage 1)

The repository does not contain true 30-day post-discharge readmissions or ICD-coded comorbidities. **Profile insight** is drawn from **Stage 1** here: demographics (`Age`, `Gender`), timing (`ICULOS`, `HospAdmTime`), ICU mapping (`Unit1`–`Unit2`), and physiology/lab trajectories. A **Random Forest** on the **same 6–12h onset target** gives nonlinear risk estimates; **Gini-based feature importances** from the forest (bar plot + CSV) summarize which inputs the model relies on most.

---

Run the next cell after the holdout models (`X_tr`, `X_te`, `y_tr`, `y_te`, `feature_names`) are defined.

In [ ]:
# Random Forest (Stage 1) + feature importance (Gini)
from sklearn.ensemble import RandomForestClassifier

rf_stage1 = RandomForestClassifier(
    n_estimators=400,
    max_depth=16,
    min_samples_leaf=15,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)
rf_stage1.fit(X_tr, y_tr)
p_rf_te = rf_stage1.predict_proba(X_te)[:, 1]
print("Stage-1 RF holdout AUROC:", round(roc_auc_score(y_te, p_rf_te), 4))
print("Stage-1 RF holdout AUPRC:", round(average_precision_score(y_te, p_rf_te), 4))

rf_imp1 = (
    pd.DataFrame({"feature": feature_names, "importance": rf_stage1.feature_importances_})
    .sort_values("importance", ascending=True)
)
k_rf = min(20, len(rf_imp1))
plot_rf = rf_imp1.tail(k_rf)
plt.figure(figsize=(9, 7))
plt.barh(plot_rf["feature"], plot_rf["importance"], color="#2ca02c")
plt.xlabel("Gini importance (mean decrease impurity)")
plt.title("Stage 1 RF: feature importance (6–12h sepsis onset)")
plt.tight_layout()
plt.show()

OUT_DIR.mkdir(parents=True, exist_ok=True)
rf_imp1.sort_values("importance", ascending=False).to_csv(OUT_DIR / "rf_feature_importance_stage1.csv", index=False)
print("Saved", OUT_DIR / "rf_feature_importance_stage1.csv")

### c(i) What-if: DSS alert threshold (Stage 1 holdout)

**Question:** How does moving the alert cutoff (trading sensitivity for specificity) change **how many alerts fire**, **how many true cases are caught**, and **workload/noise**—and, under explicit assumptions, **how much “response” capacity binds**?

**Approach:** On the **same 20% patient-level holdout** as the rest of this notebook, treat **alert = score ≥ threshold** for each patient-hour. Sweep thresholds on **RF** and **logistic** probabilities. Report **alerts per 1000 patient-hours**, confusion counts, **sensitivity / specificity / precision / NPV**, and **fatigue proxies** (`alerts_per_tp`, `fp_per_tp`). This EHR extract does **not** contain real clinician acknowledgment times, so **response rate** is **not observed**; we add a **sensitivity-analysis simulation**: reviewers can triage at most **`REVIEW_CAPACITY_PER_1000_PATIENT_HOURS`** alerts per 1000 patient-hours, always reviewing **highest score first** among firing alerts, and we report **`sensitivity_under_review_capacity`** (fraction of true onset rows that still receive a reviewed alert).

Run the next cell after the **holdout** and **RF** cells (`y_te`, `p_rf_te`, `p_logit_te`, `rf_stage1`, `final_logit`). Serialized models (`*.joblib`) are written by the **final persist** cell; they are **gitignored** if large—zip them for the LMS submission if required.

In [ ]:
# c(i) Threshold sweep + illustrative triage capacity (holdout)
REVIEW_CAPACITY_PER_1000_PATIENT_HOURS = 40.0


def threshold_curve(p, y, *, model_key: str, review_cap_per_1000: float) -> pd.DataFrame:
    p = np.asarray(p, dtype=float)
    y = np.asarray(y, dtype=int)
    n = len(y)
    n_pos = int(y.sum())
    n_neg = n - n_pos
    cap = max(1, int(review_cap_per_1000 * n / 1000.0))

    lo, hi = float(np.min(p)), float(np.max(p))
    if lo >= hi:
        thresholds = np.array([lo])
    else:
        thresholds = np.unique(np.linspace(lo, hi, 201))

    rows = []
    for t in thresholds:
        alert = p >= t
        n_alerts = int(alert.sum())
        tp = int(np.sum(alert & (y == 1)))
        fp = int(np.sum(alert & (y == 0)))
        tn = int(np.sum((~alert) & (y == 0)))
        fn = int(np.sum((~alert) & (y == 1)))

        sens = tp / n_pos if n_pos else 0.0
        spec = tn / n_neg if n_neg else 0.0
        prec = tp / n_alerts if n_alerts else 0.0
        npv = tn / (tn + fn) if (tn + fn) else 0.0
        alert_rate_per_1000 = 1000.0 * n_alerts / n

        if n_alerts == 0:
            tp_reviewed = 0
            n_reviewed = 0
        else:
            idx = np.flatnonzero(alert)
            take = idx[np.argsort(-p[idx])][: min(n_alerts, cap)]
            n_reviewed = int(len(take))
            tp_reviewed = int(y[take].sum())
        sens_cap = tp_reviewed / n_pos if n_pos else 0.0

        rows.append(
            {
                "model": model_key,
                "threshold": float(t),
                "n_holdout": n,
                "n_alerts": n_alerts,
                "alert_rate_per_1000_patient_hours": alert_rate_per_1000,
                "tp": tp,
                "fp": fp,
                "tn": tn,
                "fn": fn,
                "sensitivity_uncapped": sens,
                "specificity": spec,
                "precision": prec,
                "npv": npv,
                "alerts_per_tp": (n_alerts / tp) if tp else np.nan,
                "fp_per_tp": (fp / tp) if tp else np.nan,
                "review_capacity_per_1000": review_cap_per_1000,
                "n_alerts_reviewed_sim": n_reviewed,
                "tp_reviewed_under_capacity": tp_reviewed,
                "sensitivity_under_review_capacity": sens_cap,
            }
        )
    return pd.DataFrame(rows)


curve_rf = threshold_curve(
    p_rf_te,
    y_te,
    model_key="rf_stage1_holdout",
    review_cap_per_1000=REVIEW_CAPACITY_PER_1000_PATIENT_HOURS,
)
curve_log = threshold_curve(
    p_logit_te,
    y_te,
    model_key="logistic_holdout",
    review_cap_per_1000=REVIEW_CAPACITY_PER_1000_PATIENT_HOURS,
)
curves = pd.concat([curve_rf, curve_log], ignore_index=True)

OUT_DIR.mkdir(parents=True, exist_ok=True)
curves.to_csv(OUT_DIR / "threshold_alert_tradeoff_holdout.csv", index=False)

assumptions_doc = {
    "question": "c(i) Adjusting DSS alert threshold: alerts, simulated triage, fatigue proxies",
    "cohort": cohort_note,
    "holdout_n": int(len(y_te)),
    "holdout_positives": int(y_te.sum()),
    "scores": "RF Stage-1 `predict_proba` and holdout logistic `predict_proba` (same rows).",
    "assumption_alert_rule": "Binary alert if score >= threshold; each row is one independent patient-hour (no deduping consecutive alarms per patient).",
    "assumption_no_observed_response": "True clinician response / acknowledgment is not in the dataset; uncapped sensitivity is the upper bound if every alert were reviewed.",
    "assumption_triage_simulation": (
        f"If more than cap={int(max(1, int(REVIEW_CAPACITY_PER_1000_PATIENT_HOURS * len(y_te) / 1000.0)))} "
        "alerts fire, reviewers only examine the highest-scoring alerts up to that cap (per 1000 patient-hours basis)."
    ),
    "review_capacity_per_1000_patient_hours": REVIEW_CAPACITY_PER_1000_PATIENT_HOURS,
    "fatigue_proxies": "alerts_per_tp and fp_per_tp rise when lowering threshold: more alarms per caught case and more false alarms per true alarm.",
}
with open(OUT_DIR / "c1_threshold_whatif_assumptions.json", "w") as f:
    json.dump(assumptions_doc, f, indent=2)

# Snapshots: RF rows nearest target uncapped sensitivities
rf_c = curve_rf.reset_index(drop=True)
snapshots = []
for tgt in (0.65, 0.45, 0.30):
    j = int(np.abs(rf_c["sensitivity_uncapped"].values - tgt).argmin())
    row = rf_c.iloc[j].to_dict()
    row["target_sensitivity_approx"] = tgt
    snapshots.append(row)
with open(OUT_DIR / "c1_rf_operating_point_snapshots.json", "w") as f:
    json.dump(snapshots, f, indent=2)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
for m, lab in [("rf_stage1_holdout", "RF"), ("logistic_holdout", "Logistic")]:
    d = curves[curves["model"] == m].sort_values("threshold")
    axes[0].plot(d["threshold"], d["sensitivity_uncapped"], label=f"{lab} sens")
    axes[0].plot(d["threshold"], d["specificity"], linestyle="--", label=f"{lab} spec")
axes[0].set_xlabel("Alert threshold (prob)")
axes[0].set_ylabel("Rate")
axes[0].set_title("Sensitivity / specificity vs threshold")
axes[0].legend(fontsize=8, ncol=2)

for m, lab in [("rf_stage1_holdout", "RF"), ("logistic_holdout", "Logistic")]:
    d = curves[curves["model"] == m].sort_values("threshold")
    axes[1].plot(d["threshold"], d["alert_rate_per_1000_patient_hours"], label=f"{lab} alerts/1k")
axes[1].set_xlabel("Alert threshold (prob)")
axes[1].set_ylabel("Alerts per 1000 patient-hours")
axes[1].set_title("Alert volume vs threshold")
axes[1].legend(fontsize=8)

d = curve_rf.sort_values("threshold")
axes[2].plot(d["threshold"], d["sensitivity_uncapped"], label="RF sens (all reviewed)")
axes[2].plot(d["threshold"], d["sensitivity_under_review_capacity"], label="RF sens (cap sim)")
axes[2].set_xlabel("Alert threshold (prob)")
axes[2].set_ylabel("Sensitivity")
axes[2].set_title("RF: uncapped vs capacity-limited triage")
axes[2].legend(fontsize=8)
plt.show()

print("Saved:", OUT_DIR / "threshold_alert_tradeoff_holdout.csv")
print("Saved:", OUT_DIR / "c1_threshold_whatif_assumptions.json")
print("Saved:", OUT_DIR / "c1_rf_operating_point_snapshots.json")

### c(ii) What-if: stability when key labs are **missing** or **delayed** (Stage 1 holdout)

**Question:** If important laboratory values are **not yet in the chart** (missing) or only reflect **older draws** (delayed), how much do **RF risk scores** and **discrimination** move relative to the **reference** case where all inputs are present as in training?

**Approach (frozen Stage 1 RF):** We keep **`rf_stage1`** fixed (trained on the full feature matrix). On the **holdout** we build counterfactual matrices: (1) **missing** — set a **key-lab bundle** (six priority labs plus any same-name **rolling / delta** features in `X`) to **NaN**, then apply a **`SimpleImputer(median)` fit on the training set** before `predict_proba` (deployment-style fill from the training distribution). (2) **delayed** — for the same priority **base** labs only, replace the value at hour `ICULOS` with the patient’s own value at **`ICULOS − Δ`** hours when it exists; remaining gaps are filled by that same median imputer. We do **not** re-derive rolling features under delay (stated as a limitation in the assumptions file).

**Metrics:** holdout **AUROC / AUPRC**, **Spearman correlation** and **mean absolute change** in probability vs reference, and **overlap** of the top **10%** of rows by risk score (triage list stability).

Run after **`X_tr`, `X_te`, `y_te`, `df`, `idx_te`, `rf_stage1`** exist (holdout + RF cells).

In [ ]:
# c(ii) Missing / delayed key labs — frozen RF, holdout sensitivity analysis
from sklearn.impute import SimpleImputer

KEY_LABS_ORDERED = [
    "Lactate",
    "WBC",
    "Creatinine",
    "Platelets",
    "BUN",
    "Glucose",
    "Bilirubin_total",
    "Hgb",
    "Hct",
    "PTT",
]
KEY_LABS = [c for c in KEY_LABS_ORDERED if c in X_te.columns][:6]


def derived_feature_names(lab: str, columns: list[str]) -> list[str]:
    out = [lab]
    for suffix in ("_roll3", "_delta1"):
        c = f"{lab}{suffix}"
        if c in columns:
            out.append(c)
    return out


def missingness_columns(key_labs: list[str], columns: list[str]) -> list[str]:
    s: set[str] = set()
    for lab in key_labs:
        for c in derived_feature_names(lab, columns):
            s.add(c)
    return sorted(s)


def apply_median_impute(X_df: pd.DataFrame, imputer: SimpleImputer) -> pd.DataFrame:
    arr = imputer.transform(X_df)
    return pd.DataFrame(arr, columns=X_df.columns, index=X_df.index)


def rf_probs(X_df: pd.DataFrame, imputer: SimpleImputer) -> np.ndarray:
    Xi = apply_median_impute(X_df, imputer)
    return rf_stage1.predict_proba(Xi)[:, 1]


def top_decile_overlap(p_ref: np.ndarray, p_alt: np.ndarray, frac: float = 0.1) -> float:
    k = max(1, int(frac * len(p_ref)))
    top_ref = set(np.argsort(-p_ref)[:k])
    top_alt = set(np.argsort(-p_alt)[:k])
    return float(len(top_ref & top_alt) / k)


def delay_base_labs(
    X_te: pd.DataFrame,
    df: pd.DataFrame,
    idx_te: np.ndarray,
    key_labs: list[str],
    delay_hours: int,
) -> pd.DataFrame:
    """Same row order as X_te; only base lab columns overwritten with lagged values."""
    Xd = X_te.copy()
    df_te = df.iloc[idx_te].reset_index(drop=True)
    if not X_te.index.equals(df_te.index) and len(X_te) == len(df_te):
        df_te = df_te.reset_index(drop=True)
        Xd = Xd.reset_index(drop=True)

    for lab in key_labs:
        if lab not in Xd.columns or lab not in df_te.columns:
            continue
        delayed = np.full(len(df_te), np.nan, dtype=float)
        for _, grp in df_te.groupby("patient_id", sort=False):
            pos = grp.index.to_numpy()
            los = grp["ICULOS"].to_numpy(dtype=float)
            vals = grp[lab].to_numpy(dtype=float)
            lookup: dict[int, float] = {}
            for lo, v in zip(los, vals):
                if pd.notna(v):
                    lookup[int(lo)] = float(v)
            for i, lo in zip(pos, los):
                delayed[i] = lookup.get(int(lo) - delay_hours, np.nan)
        Xd[lab] = delayed
    return Xd


def spearman_vs_ref(p_ref: np.ndarray, p_alt: np.ndarray) -> float:
    ra = pd.Series(p_ref).rank(method="average")
    rb = pd.Series(p_alt).rank(method="average")
    if float(ra.std()) < 1e-12 or float(rb.std()) < 1e-12:
        return 1.0 if np.allclose(p_ref, p_alt, atol=1e-6) else 0.0
    return float(np.corrcoef(ra.to_numpy(), rb.to_numpy())[0, 1])


def scenario_metrics(y: np.ndarray, p_ref: np.ndarray, p_alt: np.ndarray, name: str) -> dict:
    return {
        "scenario": name,
        "holdout_auroc": float(roc_auc_score(y, p_alt)),
        "holdout_auprc": float(average_precision_score(y, p_alt)),
        "spearman_vs_reference": spearman_vs_ref(p_ref, p_alt),
        "mean_abs_prob_delta": float(np.mean(np.abs(p_alt - p_ref))),
        "median_abs_prob_delta": float(np.median(np.abs(p_alt - p_ref))),
        "top_decile_overlap_vs_reference": top_decile_overlap(p_ref, p_alt),
    }


imputer = SimpleImputer(strategy="median")
imputer.fit(X_tr)

X_te_use = X_te.reset_index(drop=True)
p_ref = rf_probs(X_te_use, imputer)

miss_cols = missingness_columns(KEY_LABS, list(X_te_use.columns))
X_miss = X_te_use.copy()
for c in miss_cols:
    X_miss[c] = np.nan
p_miss = rf_probs(X_miss, imputer)

rows = [scenario_metrics(y_te, p_ref, p_ref, "reference_full_imputer_path")]
rows.append(scenario_metrics(y_te, p_ref, p_miss, "key_labs_missing_median_impute"))

for dh in (2, 4):
    X_del = delay_base_labs(X_te_use, df, idx_te, KEY_LABS, delay_hours=dh)
    p_del = rf_probs(X_del, imputer)
    rows.append(scenario_metrics(y_te, p_ref, p_del, f"key_labs_delayed_{dh}h_median_impute"))

# Single-lab missing (base + derived) for extra granularity
for lab in KEY_LABS:
    cols = derived_feature_names(lab, list(X_te_use.columns))
    if not cols:
        continue
    Xm = X_te_use.copy()
    for c in cols:
        Xm[c] = np.nan
    pm = rf_probs(Xm, imputer)
    rows.append(scenario_metrics(y_te, p_ref, pm, f"missing_only_{lab}"))

c2_df = pd.DataFrame(rows)
OUT_DIR.mkdir(parents=True, exist_ok=True)
c2_df.to_csv(OUT_DIR / "c2_lab_missing_delay_robustness.csv", index=False)

c2_assumptions = {
    "question": "c(ii) Prediction stability when key labs are missing or delayed",
    "cohort": cohort_note,
    "model": "Frozen rf_stage1 trained on full inputs; holdout counterfactuals only.",
    "key_labs_used": KEY_LABS,
    "missingness_columns_masked": miss_cols,
    "imputation_assumption": "SimpleImputer(median) fit on X_tr only; applied before predict_proba (same column order as X).",
    "delay_assumption": "Per patient_id, base lab at ICULOS replaced with value at ICULOS - delta_hours when present; else NaN then median imputed.",
    "limitation_roll_features": "Rolling/delta lab features are not recomputed under delay-only scenarios (only base labs lagged).",
}
with open(OUT_DIR / "c2_lab_robustness_assumptions.json", "w") as f:
    json.dump(c2_assumptions, f, indent=2)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axes[0].hexbin(p_ref, p_miss, gridsize=40, cmap="Blues", mincnt=1)
axes[0].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.6)
axes[0].set_xlabel("RF prob (reference)")
axes[0].set_ylabel("RF prob (key labs missing)")
axes[0].set_title("Holdout scores: missing labs")

p_del2 = rf_probs(delay_base_labs(X_te_use, df, idx_te, KEY_LABS, 2), imputer)
axes[1].hexbin(p_ref, p_del2, gridsize=40, cmap="Greens", mincnt=1)
axes[1].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.6)
axes[1].set_xlabel("RF prob (reference)")
axes[1].set_ylabel("RF prob (labs delayed 2h)")
axes[1].set_title("Holdout scores: 2h lab delay")
plt.show()

print("Key labs:", KEY_LABS)
print(c2_df.to_string(index=False))
print("Saved:", OUT_DIR / "c2_lab_missing_delay_robustness.csv")
print("Saved:", OUT_DIR / "c2_lab_robustness_assumptions.json")

### c(iii) What-if: **vitals-only** (and **no-lab**) vs **full** features (Stage 1 holdout)

**Question:** How well can we predict **6–12 h sepsis onset** using **vital signs before chemistry is available**, and how does that compare to the **full** model that includes labs, derived lab trajectories, and other inputs?

**Approach:** On the **same patient-level holdout** as Stage 1, we **retrain separate Random Forests** (same hyperparameters as `rf_stage1`) on **restricted feature sets** and compare **holdout AUROC, AUPRC**, **Spearman correlation** of predicted probabilities vs the **full** model, **mean |Δp|**, and **top-decile overlap** (who stays in the highest-risk decile). **Sensitivity-analysis scenarios:** (1) **vitals-only** — HR, MAP, O2Sat, Temp, Resp, SBP, DBP, SaO2, FiO2, EtCO2 plus their `_roll3` / `_delta1` columns if present in `X`; (2) **vitals + ICULOS** — adds only the ICU clock; (3) **non-lab features** — all columns whose **base name** (strip `_roll3` / `_delta1`) is **not** in a fixed **laboratory stem set** (vitals + demographics + units + timing, **no** chemistry / hematology / ABG labs); (4) **full** — existing `rf_stage1` scores (`p_rf_te`). This separates **strict vitals** from a **“labs not yet in chart”** proxy that still allows **context** variables.

Run after **`X_tr`, `X_te`, `y_te`, `feature_names`, `rf_stage1`, `p_rf_te`** are defined.

In [ ]:
# c(iii) Vitals-only / no-lab vs full — retrained RF, same holdout (sensitivity analysis)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import RocCurveDisplay

VITAL_BASES = frozenset({"HR", "MAP", "O2Sat", "Temp", "Resp", "SBP", "DBP", "SaO2", "FiO2", "EtCO2"})

LAB_STEMS = frozenset({
    "WBC",
    "Lactate",
    "Creatinine",
    "Glucose",
    "Platelets",
    "BUN",
    "Bilirubin_total",
    "Bilirubin_direct",
    "Hgb",
    "Hct",
    "PTT",
    "Calcium",
    "Phosphate",
    "Potassium",
    "Chloride",
    "HCO3",
    "pH",
    "PaCO2",
    "Magnesium",
    "AST",
    "Alkalinephos",
    "BaseExcess",
    "Fibrinogen",
    "TroponinI",
})


def stem_feature(c: str) -> str:
    for suf in ("_roll3", "_delta1"):
        if c.endswith(suf):
            return c[: -len(suf)]
    return c


def vital_column_names(feature_names: list[str]) -> list[str]:
    out: list[str] = []
    for c in feature_names:
        if c in VITAL_BASES:
            out.append(c)
            continue
        for b in VITAL_BASES:
            if c == f"{b}_roll3" or c == f"{b}_delta1":
                out.append(c)
                break
    return out


def non_lab_column_names(feature_names: list[str]) -> list[str]:
    return [c for c in feature_names if stem_feature(c) not in LAB_STEMS]


def top_decile_overlap(p_ref: np.ndarray, p_alt: np.ndarray, frac: float = 0.1) -> float:
    k = max(1, int(frac * len(p_ref)))
    top_ref = set(np.argsort(-p_ref)[:k])
    top_alt = set(np.argsort(-p_alt)[:k])
    return float(len(top_ref & top_alt) / k)


def spearman_vs_ref(p_ref: np.ndarray, p_alt: np.ndarray) -> float:
    ra = pd.Series(p_ref).rank(method="average")
    rb = pd.Series(p_alt).rank(method="average")
    if float(ra.std()) < 1e-12 or float(rb.std()) < 1e-12:
        return 1.0 if np.allclose(p_ref, p_alt, atol=1e-6) else 0.0
    return float(np.corrcoef(ra.to_numpy(), rb.to_numpy())[0, 1])


def eval_vs_full(name: str, cols: list[str], p_full: np.ndarray, y: np.ndarray, p_alt: np.ndarray) -> dict:
    return {
        "scenario": name,
        "n_features": len(cols),
        "holdout_auroc": float(roc_auc_score(y, p_alt)),
        "holdout_auprc": float(average_precision_score(y, p_alt)),
        "spearman_vs_full_model": spearman_vs_ref(p_full, p_alt),
        "mean_abs_prob_delta_vs_full": float(np.mean(np.abs(p_alt - p_full))),
        "median_abs_prob_delta_vs_full": float(np.median(np.abs(p_alt - p_full))),
        "top_decile_overlap_vs_full": top_decile_overlap(p_full, p_alt),
        "delta_auroc_vs_full": float(roc_auc_score(y, p_alt) - roc_auc_score(y, p_full)),
    }


RF_KW = dict(
    n_estimators=400,
    max_depth=16,
    min_samples_leaf=15,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)

vcols = vital_column_names(feature_names)
nlcols = non_lab_column_names(feature_names)
if "ICULOS" in feature_names:
    v_iculos = ["ICULOS"] + [c for c in vcols if c != "ICULOS"]
else:
    v_iculos = list(vcols)

rows = []
p_full = np.asarray(p_rf_te, dtype=float)
y_h = np.asarray(y_te, dtype=int)

rows.append(
    {
        "scenario": "full_all_features",
        "n_features": len(feature_names),
        "holdout_auroc": float(roc_auc_score(y_h, p_full)),
        "holdout_auprc": float(average_precision_score(y_h, p_full)),
        "spearman_vs_full_model": 1.0,
        "mean_abs_prob_delta_vs_full": 0.0,
        "median_abs_prob_delta_vs_full": 0.0,
        "top_decile_overlap_vs_full": 1.0,
        "delta_auroc_vs_full": 0.0,
    }
)

trained = {}
probs: dict[str, np.ndarray] = {"full_all_features": p_full}

for name, cols in [
    ("vitals_only", vcols),
    ("vitals_plus_ICULOS", v_iculos),
    ("non_lab_features", nlcols),
]:
    if len(cols) < 2:
        continue
    rf_c = RandomForestClassifier(**RF_KW)
    rf_c.fit(X_tr[cols], y_tr)
    p_alt = rf_c.predict_proba(X_te[cols])[:, 1]
    trained[name] = rf_c
    probs[name] = p_alt
    rows.append(eval_vs_full(name, cols, p_full, y_h, p_alt))

c3_df = pd.DataFrame(rows)
OUT_DIR.mkdir(parents=True, exist_ok=True)
c3_df.to_csv(OUT_DIR / "c3_vitals_vs_full_holdout.csv", index=False)

c3_assumptions = {
    "question": "c(iii) Vitals-only / no-lab vs full feature RF performance",
    "cohort": cohort_note,
    "holdout_n": int(len(y_h)),
    "full_model": "rf_stage1 trained on all columns in feature_names",
    "vitals_only_definition": "HR, MAP, O2Sat, Temp, Resp, SBP, DBP, SaO2, FiO2, EtCO2 plus matching _roll3 and _delta1 columns if present.",
    "non_lab_definition": "Columns whose stem (strip _roll3/_delta1) is not in LAB_STEMS (chemistry/hematology/ABG-style labs).",
    "sensitivity_note": "Same RF hyperparameters and same GroupShuffleSplit holdout as Stage 1; only the input column mask changes.",
    "n_vitals_only_features": len(vcols),
    "n_non_lab_features": len(nlcols),
}
with open(OUT_DIR / "c3_vitals_vs_full_assumptions.json", "w") as f:
    json.dump(c3_assumptions, f, indent=2)

if "vitals_only" in trained:
    imp_v = (
        pd.DataFrame({"feature": vcols, "importance": trained["vitals_only"].feature_importances_})
        .sort_values("importance", ascending=False)
    )
    imp_v.to_csv(OUT_DIR / "c3_rf_feature_importance_vitals_only.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
RocCurveDisplay.from_predictions(y_h, p_full, ax=axes[0], name=f"Full (n={len(feature_names)})", plot_chance_level=True)
if "vitals_only" in probs:
    RocCurveDisplay.from_predictions(y_h, probs["vitals_only"], ax=axes[0], name=f"Vitals-only (n={len(vcols)})")
axes[0].set_title("Holdout ROC: full vs vitals-only")

if "vitals_only" in probs:
    axes[1].hexbin(p_full, probs["vitals_only"], gridsize=45, cmap="Purples", mincnt=1)
    axes[1].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
    axes[1].set_xlabel("RF prob (full features)")
    axes[1].set_ylabel("RF prob (vitals-only)")
    axes[1].set_title("Probability calibration vs full (hexbin)")
plt.show()

print(c3_df.to_string(index=False))
print("Saved:", OUT_DIR / "c3_vitals_vs_full_holdout.csv")
print("Saved:", OUT_DIR / "c3_vitals_vs_full_assumptions.json")
if "vitals_only" in trained:
    print("Saved:", OUT_DIR / "c3_rf_feature_importance_vitals_only.csv")

In [ ]:
# Persist results for assignment artifacts
OUT_DIR.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(OUT_DIR / "cv_metrics_by_fold.csv", index=False)
logit_imp.to_csv(OUT_DIR / "logistic_coefficient_magnitude_cv.csv", index=False)
perm_imp.to_csv(OUT_DIR / "permutation_importance_hist_gbrt_holdout.csv", index=False)

summary = {
    "task": "Binary onset of sepsis in hours t+6..t+12",
    "cohort": cohort_note,
    "n_rows": int(len(df)),
    "positive_rate": float(y.mean()),
    "holdout_auroc_logistic": float(roc_auc_score(y_te, p_logit_te)),
    "holdout_auprc_logistic": float(average_precision_score(y_te, p_logit_te)),
    "holdout_auroc_hist_gbrt": float(roc_auc_score(y_te, p_hgb_te)),
    "holdout_auprc_hist_gbrt": float(average_precision_score(y_te, p_hgb_te)),
    "holdout_report_hist_gbrt": classification_report(y_te, hgb_final.predict(X_te), digits=4, output_dict=True),
}
try:
    summary["holdout_auroc_rf_stage1"] = float(roc_auc_score(y_te, p_rf_te))
    summary["holdout_auprc_rf_stage1"] = float(average_precision_score(y_te, p_rf_te))
except NameError:
    pass

summary["c1_alert_threshold_whatif_artifacts"] = {
    "description": "c(i) threshold sweep on holdout; see c1_threshold_whatif_assumptions.json for explicit assumptions",
    "csv": "threshold_alert_tradeoff_holdout.csv",
    "assumptions_json": "c1_threshold_whatif_assumptions.json",
    "rf_snapshots_json": "c1_rf_operating_point_snapshots.json",
    "serialized_models_joblib": ["rf_stage1_holdout.joblib", "logistic_stage1_holdout.joblib"],
}
summary["c2_lab_missing_delay_artifacts"] = {
    "description": "c(ii) frozen RF holdout stability under missing/delayed key labs",
    "csv": "c2_lab_missing_delay_robustness.csv",
    "assumptions_json": "c2_lab_robustness_assumptions.json",
}
summary["c3_vitals_vs_full_artifacts"] = {
    "description": "c(iii) retrained RF on vitals-only / non-lab / full; same holdout",
    "csv": "c3_vitals_vs_full_holdout.csv",
    "assumptions_json": "c3_vitals_vs_full_assumptions.json",
    "vitals_only_importance_csv": "c3_rf_feature_importance_vitals_only.csv",
}

c3_csv = OUT_DIR / "c3_vitals_vs_full_holdout.csv"
if c3_csv.exists():
    c3tab = pd.read_csv(c3_csv)
    for scen, auroc_k, auprc_k in [
        ("vitals_only", "holdout_auroc_rf_vitals_only", "holdout_auprc_rf_vitals_only"),
        ("non_lab_features", "holdout_auroc_rf_non_lab_features", "holdout_auprc_rf_non_lab_features"),
        ("vitals_plus_ICULOS", "holdout_auroc_rf_vitals_plus_iculos", "holdout_auprc_rf_vitals_plus_iculos"),
    ]:
        sub = c3tab[c3tab["scenario"] == scen]
        if len(sub):
            summary[auroc_k] = float(sub.iloc[0]["holdout_auroc"])
            summary[auprc_k] = float(sub.iloc[0]["holdout_auprc"])

with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(summary, f, indent=2)

try:
    import joblib

    joblib.dump(rf_stage1, OUT_DIR / "rf_stage1_holdout.joblib")
    joblib.dump(final_logit, OUT_DIR / "logistic_stage1_holdout.joblib")
    print("Saved joblib models to", OUT_DIR)
except NameError:
    pass

print("Saved outputs to", OUT_DIR)